# Trendlines with Breaks [LuxAlgo] 概念實作與多標的策略回測

本 notebook 依照期中作業要求，實作一個參考 **Trendlines with Breaks [LuxAlgo]** 概念的技術指標策略。原指標重點是使用 pivot point 建立趨勢線，並標示價格突破趨勢線的訊號。LuxAlgo 官方說明指出，該指標以 pivot-based trendlines 尋找 breakout，並可使用 ATR、標準差或線性迴歸作為趨勢線斜率計算方式。

參考來源：

- LuxAlgo indicator library: https://www.luxalgo.com/library/indicator/trendlines-with-breaks
- TradingView open-source indicator page: https://www.tradingview.com/script/IYL88A1N-Trendlines-with-Breaks-LuxAlgo/

本作業不是直接複製 Pine Script，而是以 Python 重新建立可回測版本。為避免未來函數問題，pivot high/low 必須等右側 `length` 根 K 線完成後才確認，因此策略訊號會自然延遲。

## 1. 套件與研究設定

In [ ]:
!pip -q install yfinance

import warnings
warnings.filterwarnings("ignore")

import os
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

DATA_PERIOD = "max"
START_DATE = None
END_DATE = None
ANALYSIS_START = "2000-01-01"
ANALYSIS_END = None
CACHE_DIR = "price_cache"

TICKERS = ["TSM", "AAPL", "NVDA", "MSFT", "AMD", "SPY", "QQQ"]
MAIN_TICKER = "TSM"

TRANSACTION_COST = 0.001425
STOP_LOSS = 0.12
TAKE_PROFIT = None
INTERVAL = "1d" # Switched back to daily

## 2. 資料抓取與預處理

使用 `yfinance` 抓取日資料，並建立本地快取 CSV，避免重複下載造成 Yahoo Finance 限流。回測使用調整後收盤價 `Adj Close` 作為交易價格，以反映拆股與股利調整。

In [ ]:
def cache_file_path(ticker, period=DATA_PERIOD, start=START_DATE, end=END_DATE, interval="1d"):
    cache_dir = Path(CACHE_DIR)
    cache_dir.mkdir(parents=True, exist_ok=True)
    if period == "max":
        range_tag = "MAX"
    else:
        range_tag = f"{start}_{end}"
    return cache_dir / f"{ticker}_{range_tag}_{interval}.csv"


def normalize_price_frame(df):
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df.reset_index() if "Date" not in df.columns and "Datetime" not in df.columns else df.copy()
    if "Datetime" in df.columns and "Date" not in df.columns:
        df = df.rename(columns={"Datetime": "Date"})
    return df


def validate_price_frame(df, ticker):
    required = {"Date", "Open", "High", "Low", "Close", "Volume"}
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f"Cache/data for {ticker} is missing columns: {missing}")
    if "Adj Close" not in df.columns:
        df["Adj Close"] = df["Close"]
    return df


def fetch_from_yfinance(ticker, period=DATA_PERIOD, start=START_DATE, end=END_DATE, interval="1d"):
    try:
        if period == "max":
            df = yf.download(
                ticker,
                period="max",
                interval=interval,
                auto_adjust=False,
                progress=False,
                threads=False,
            )
        else:
            df = yf.download(
                ticker,
                start=start,
                end=end,
                interval=interval,
                auto_adjust=False,
                progress=False,
                threads=False,
            )
    except Exception as err:
        print(f"yfinance download failed for {ticker} ({interval}): {err}")
        df = pd.DataFrame()

    if df.empty and period != "max" and interval == "1d":
        try:
            print(f"Trying period='max' fallback for {ticker}")
            df = yf.download(
                ticker,
                period="max",
                interval="1d",
                auto_adjust=False,
                progress=False,
                threads=False,
            )
            if not df.empty and start is not None and end is not None:
                df = df.loc[(df.index >= pd.to_datetime(start)) & (df.index <= pd.to_datetime(end))]
        except Exception as err:
            print(f"period='max' fallback failed for {ticker}: {err}")
            df = pd.DataFrame()

    if df.empty:
        raise ValueError(
            f"Unable to download {ticker} ({interval}). Yahoo Finance may be rate-limiting this environment. "
            "Retry later, run in Colab, or provide the matching CSV in price_cache/."
        )
    return normalize_price_frame(df)


def download_price_data(ticker, period=DATA_PERIOD, start=START_DATE, end=END_DATE, use_cache=True, interval="1d"):
    cache_path = cache_file_path(ticker, period, start, end, interval)

    if use_cache and cache_path.exists():
        try:
            df = pd.read_csv(cache_path)
            df = validate_price_frame(df, ticker)
            print(f"Using cached data: {ticker} ({interval}) -> {cache_path}")
        except Exception as err:
            print(f"Cache invalid for {ticker}; re-downloading. Reason: {err}")
            df = fetch_from_yfinance(ticker, period, start, end, interval)
            df = validate_price_frame(df, ticker)
            df.to_csv(cache_path, index=False)
            print(f"Downloaded and refreshed cache: {ticker} ({interval}) -> {cache_path}")
    else:
        df = fetch_from_yfinance(ticker, period, start, end, interval)
        df = validate_price_frame(df, ticker)
        df.to_csv(cache_path, index=False)
        print(f"Downloaded and cached: {ticker} ({interval}) -> {cache_path}")

    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").drop_duplicates("Date").set_index("Date")
    if ANALYSIS_START is not None:
        df = df.loc[pd.to_datetime(ANALYSIS_START):]
    if ANALYSIS_END is not None:
        df = df.loc[:pd.to_datetime(ANALYSIS_END)]
    price_col = "Adj Close" if "Adj Close" in df.columns else "Close"
    df["Price"] = df[price_col]
    df = df.dropna(subset=["Open", "High", "Low", "Close", "Price", "Volume"])
    df["Return"] = df["Price"].pct_change()
    return df.dropna(subset=["Return"])

sample = download_price_data(MAIN_TICKER, interval=INTERVAL)
display(sample.head())
print(f"{MAIN_TICKER} ({INTERVAL}) loaded, rows: {len(sample):,}")

## 3. 指標邏輯說明

本 notebook 將 Trendlines with Breaks 的概念拆成三步：

1. **Pivot 偵測**：若某日 high 是左右各 `length` 日內最高，則為 pivot high；若 low 是左右各 `length` 日內最低，則為 pivot low。因為需要右側資料確認，所以 pivot 會在 `length` 日後才可用。
2. **趨勢線延伸**：pivot high 形成上方壓力線，pivot low 形成下方支撐線。斜率可用 ATR、標準差或線性迴歸估計。
3. **Breakout 訊號**：價格突破上方趨勢線視為多方突破；跌破下方趨勢線視為弱勢或出場訊號。

這裡的實作重點是回測可信度，因此所有 pivot 都只在確認後才加入訊號計算。

In [ ]:
def add_atr(df, window=14):
    out = df.copy()
    prev_close = out["Close"].shift(1)
    tr = pd.concat([
        out["High"] - out["Low"],
        (out["High"] - prev_close).abs(),
        (out["Low"] - prev_close).abs(),
    ], axis=1).max(axis=1)
    out["ATR"] = tr.rolling(window).mean()
    return out

def rolling_linreg_slope(series, window):
    x = np.arange(window)
    x_centered = x - x.mean()
    denominator = (x_centered ** 2).sum()

    def slope(values):
        if np.isnan(values).any():
            return np.nan
        y = values - values.mean()
        return float((x_centered * y).sum() / denominator)

    return series.rolling(window).apply(slope, raw=True)

def slope_series(df, length, method="atr", mult=1.0):
    if method == "atr":
        return df["ATR"] / length * mult
    if method == "stdev":
        return df["Price"].rolling(length).std() / length * mult
    if method == "linreg":
        return rolling_linreg_slope(df["Price"], length).abs() * mult
    raise ValueError("method must be atr, stdev, or linreg")

def add_trendline_breaks(df, length=14, slope_method="atr", slope_mult=1.0):
    out = add_atr(df, window=max(14, length)).copy()
    slope = slope_series(out, length, slope_method, slope_mult).fillna(0)
    n = len(out)

    pivot_high_confirmed = np.zeros(n, dtype=bool)
    pivot_low_confirmed = np.zeros(n, dtype=bool)

    highs = out["High"].to_numpy()
    lows = out["Low"].to_numpy()
    for i in range(length, n - length):
        high_window = highs[i - length:i + length + 1]
        low_window = lows[i - length:i + length + 1]
        if highs[i] == np.nanmax(high_window):
            pivot_high_confirmed[i + length] = True
        if lows[i] == np.nanmin(low_window):
            pivot_low_confirmed[i + length] = True

    upper = np.full(n, np.nan)
    lower = np.full(n, np.nan)
    current_upper = np.nan
    current_lower = np.nan
    current_upper_slope = 0.0
    current_lower_slope = 0.0

    for i in range(n):
        if pivot_high_confirmed[i]:
            pivot_index = i - length
            current_upper = highs[pivot_index] - slope.iloc[i] * length
            current_upper_slope = slope.iloc[i]
        elif not np.isnan(current_upper):
            current_upper -= current_upper_slope

        if pivot_low_confirmed[i]:
            pivot_index = i - length
            current_lower = lows[pivot_index] + slope.iloc[i] * length
            current_lower_slope = slope.iloc[i]
        elif not np.isnan(current_lower):
            current_lower += current_lower_slope

        upper[i] = current_upper
        lower[i] = current_lower

    out["Upper_Trendline"] = upper
    out["Lower_Trendline"] = lower
    out["Pivot_High_Confirmed"] = pivot_high_confirmed
    out["Pivot_Low_Confirmed"] = pivot_low_confirmed
    out["Upper_Break"] = (out["Close"] > out["Upper_Trendline"]) & (out["Close"].shift(1) <= out["Upper_Trendline"].shift(1))
    out["Lower_Break"] = (out["Close"] < out["Lower_Trendline"]) & (out["Close"].shift(1) >= out["Lower_Trendline"].shift(1))
    return out

## 4. 策略設計：波動率調節與 ATR 停損

本策略目前的設計重點在於**適應性 (Adaptability)**，主要邏輯如下：

1.  **波動率調節長度 (Vol-Adjusted Length)**：
    *   根據標的的年化波動率動態調整 Pivot 偵測窗口。當波動率高時，縮短長度以提高靈敏度；波動率低時，拉長長度以過濾雜訊。
    *   公式參考：`adj_length = base_length * (0.2 / annual_vol)`。

2.  **ATR 動態停損與百分比 Trailing Stop**：
    *   ATR 停損採 `Price - (ATR * Multiplier)`，讓停損距離隨市場波動自動調整。
    *   百分比 trailing stop 則使用 `最高價 * (1 - trailing_stop_pct)`，當價格創新高時停損會跟著上移，用來鎖住部分浮盈。

3.  **放寬出場條件**：
    *   原本只要出現 `Lower_Break` 就出場，容易被短暫假跌破洗出場。
    *   新版加入 `exit_confirm_days`，可測試連續 1、2 或 3 天確認跌破後才出場。

4.  **進出場規則**：
    *   **進場**：價格收盤突破上方趨勢線 (`Upper_Break`)。
    *   **出場/停損**：連續跌破下方趨勢線達確認天數，或觸發 ATR / trailing stop。

策略維持 Long-only 且包含交易成本計算。

In [ ]:
def backtest_breakout_strategy(
    df,
    stop_mode="basic",
    stop_loss=STOP_LOSS,
    breakeven_trigger=0.05,
    transaction_cost=TRANSACTION_COST,
):
    out = df.copy()

    units = 0
    entry_price = np.nan
    highest_price_since_entry = np.nan
    stop_price = np.nan
    breakeven_activated = False

    positions = []
    actions = []
    stop_prices = []
    breakeven_flags = []

    for i in range(len(out)):
        if i == 0:
            positions.append(0)
            actions.append("Hold")
            stop_prices.append(np.nan)
            breakeven_flags.append(False)
            continue

        price = out["Price"].iloc[i]
        upper_break = bool(out["Upper_Break"].iloc[i - 1])
        lower_break = bool(out["Lower_Break"].iloc[i - 1])
        action = "Hold"

        if units == 0:
            if upper_break:
                units = 1
                entry_price = price
                highest_price_since_entry = price
                stop_price = entry_price * (1 - stop_loss)
                breakeven_activated = False
                action = "Buy"
        else:
            highest_price_since_entry = max(highest_price_since_entry, price)

            if stop_mode == "breakeven" and not breakeven_activated:
                if highest_price_since_entry >= entry_price * (1 + breakeven_trigger):
                    stop_price = entry_price
                    breakeven_activated = True

            stopped = price <= stop_price
            if lower_break or stopped:
                if stopped and breakeven_activated:
                    action = "Breakeven Stop"
                elif stopped:
                    action = "Stop Loss"
                else:
                    action = "Sell"
                units = 0
                entry_price = np.nan
                highest_price_since_entry = np.nan
                stop_price = np.nan
                breakeven_activated = False

        positions.append(units)
        actions.append(action)
        stop_prices.append(stop_price)
        breakeven_flags.append(breakeven_activated)

    out["Position"] = positions
    out["Action"] = actions
    out["Stop_Price"] = stop_prices
    out["Breakeven_Activated"] = breakeven_flags
    out["Trade"] = out["Position"].diff().abs().fillna(out["Position"].abs())
    out["Strategy_Return_Gross"] = out["Position"].shift(1).fillna(0) * out["Return"]
    out["Transaction_Cost"] = out["Trade"] * transaction_cost
    out["Strategy_Return"] = out["Strategy_Return_Gross"] - out["Transaction_Cost"]
    out["Strategy_Cum"] = (1 + out["Strategy_Return"]).cumprod()
    out["BuyHold_Cum"] = (1 + out["Return"]).cumprod()
    return out

def max_drawdown(cumulative_return):
    if cumulative_return.empty:
        return 0
    running_max = cumulative_return.cummax()
    drawdown = cumulative_return / running_max - 1
    return drawdown.min()

def performance_metrics(df, return_col="Strategy_Return", cumulative_col="Strategy_Cum"):
    if return_col not in df.columns or cumulative_col not in df.columns:
        return {"Total Return": 0, "Annual Return": 0, "Annual Volatility": 0, "Sharpe": 0, "Max Drawdown": 0, "Win Rate": 0}

    daily_ret = df[return_col].dropna()
    total_return = df[cumulative_col].iloc[-1] - 1
    years = max(len(daily_ret) / 252, 0.001)
    annual_return = (1 + total_return) ** (1 / years) - 1
    annual_vol = daily_ret.std() * np.sqrt(252)
    sharpe = annual_return / annual_vol if annual_vol > 0 else 0
    return {
        "Total Return": total_return,
        "Annual Return": annual_return,
        "Annual Volatility": annual_vol,
        "Sharpe": sharpe,
        "Max Drawdown": max_drawdown(df[cumulative_col]),
        "Win Rate": (daily_ret > 0).mean() if len(daily_ret) > 0 else 0,
    }

## 5. Strategy Parameter Search

This version makes the base parameters data-driven. Each ticker receives its own base `Length` and base `ATR Mult` from rolling past-only asset statistics, not from manual per-ticker optimization.

1. **Asset profile**: past-only rolling annualized volatility and trend efficiency are computed for every ticker.
2. **Data-driven base parameters**: high-volatility assets receive longer base breakout windows and wider ATR stops. Assets with cleaner directional trend efficiency can use a slightly shorter window.
3. **Global scale grid**: the search only tests common `Length Scale` and `ATR Scale` values, plus exit confirmation, stop type, and optional dynamic volatility.
4. **Unified final rule**: the selected scale, stop type, and dynamic-vol setting are applied to all tickers.
5. **Market-hold comparison**: all metrics are reported beside MH over the same period.

In [ ]:
import numpy as np
import pandas as pd
from itertools import product

# Unified data-driven strategy:
# each ticker gets base parameters from observable asset statistics, then a small global grid
# searches only common scaling, exit confirmation, stop type, and optional time-varying volatility adjustment.
TRENDLINE_CACHE = {}


def add_no_lookahead_parameter_series(
    df,
    lookback_days=252 * 5,
    base_length_anchor=20,
    base_atr_anchor=7.5,
    vol_target=0.275,
    trend_target=0.05,
    length_clip=(10, 36),
    atr_clip=(5.0, 13.0),
):
    """Create base parameters from information available before each trading day."""
    out = df.copy()
    prev_return = out["Return"].shift(1)
    prev_log_price = np.log(out["Price"]).shift(1)

    annual_vol = prev_return.rolling(lookback_days, min_periods=126).std() * np.sqrt(252)
    path_move = prev_log_price.diff().abs().rolling(lookback_days, min_periods=126).sum()
    net_move = (prev_log_price - prev_log_price.shift(lookback_days)).abs()
    trend_efficiency = (net_move / path_move).replace([np.inf, -np.inf], np.nan)

    annual_vol = annual_vol.fillna(vol_target)
    trend_efficiency = trend_efficiency.fillna(trend_target)

    vol_ratio = (annual_vol / vol_target).replace([np.inf, -np.inf], np.nan).fillna(1.0)
    trend_ratio = (trend_efficiency / trend_target).replace([np.inf, -np.inf], np.nan).fillna(1.0)
    trend_ratio = trend_ratio.clip(0.50, 1.80)

    length_mult = ((vol_ratio ** 0.65) / (trend_ratio ** 0.35)).clip(0.65, 1.80)
    atr_mult_factor = (vol_ratio ** 0.75).clip(0.70, 1.75)

    out["Annual_Vol_Past"] = annual_vol
    out["Trend_Efficiency_Past"] = trend_efficiency
    out["Data_Base_Length"] = (base_length_anchor * length_mult).round().clip(*length_clip).astype(int)
    out["Data_Base_ATR_Mult"] = (base_atr_anchor * atr_mult_factor).clip(*atr_clip)
    return out


def compute_asset_parameter_snapshot(price_data_by_ticker):
    rows = []
    profiles = {}
    for ticker, df in price_data_by_ticker.items():
        with_params = add_no_lookahead_parameter_series(df)
        last = with_params.iloc[-1]
        profile = {
            "Annual Vol": float(last["Annual_Vol_Past"]),
            "Trend Efficiency": float(last["Trend_Efficiency_Past"]),
            "Data Base Length": int(last["Data_Base_Length"]),
            "Data Base ATR Mult": float(last["Data_Base_ATR_Mult"]),
        }
        profiles[ticker] = profile
        rows.append({"Ticker": ticker, **profile})
    table = pd.DataFrame(rows).sort_values("Data Base Length", ascending=False)
    return profiles, table


def backtest_vol_strategy_v2(
    df,
    base_length,
    base_atr_mult,
    asset_vol_mult=1.0,
    use_dynamic_vol=True,
    dynamic_vol_power=0.5,
    slope_method="stdev",
    exit_confirm_days=1,
    stop_type="atr",
    trailing_stop_pct=0.10,
):
    out = add_no_lookahead_parameter_series(df)
    out["Rolling_Vol_30"] = out["Return"].shift(1).rolling(30).std() * np.sqrt(252)
    out["Rolling_Vol_252"] = out["Return"].shift(1).rolling(252).std() * np.sqrt(252)
    time_vol_mult = (out["Rolling_Vol_30"] / out["Rolling_Vol_252"]).replace([np.inf, -np.inf], np.nan).fillna(1.0)
    out["Time_Vol_Mult"] = np.clip(time_vol_mult, 0.70, 1.60)

    min_mult = 0.60
    max_mult = 2.20
    length_options = [8, 10, 12, 16, 20, 24, 30, 36, 44, 55, 70]
    cache_key = (id(df), slope_method, tuple(length_options))
    if cache_key not in TRENDLINE_CACHE:
        TRENDLINE_CACHE[cache_key] = {
            l: add_trendline_breaks(df, length=l, slope_method=slope_method)
            for l in length_options
        }
    tl_data = TRENDLINE_CACHE[cache_key]

    out = add_atr(out)
    n = len(out)
    units, entry_price, stop_price = 0, np.nan, np.nan
    highest_price = np.nan
    lower_break_streak = 0
    positions, actions, stop_prices = [], [], []
    upper_tl, lower_tl, effective_lengths, effective_atr_mults = [], [], [], []

    for i in range(n):
        if use_dynamic_vol:
            final_mult = asset_vol_mult * (out["Time_Vol_Mult"].iloc[i] ** dynamic_vol_power)
        else:
            final_mult = asset_vol_mult
        final_mult = float(np.clip(final_mult, min_mult, max_mult))

        data_base_length = out["Data_Base_Length"].iloc[i] * base_length
        data_base_atr = out["Data_Base_ATR_Mult"].iloc[i] * base_atr_mult
        target_l = max(5, int(round(data_base_length * final_mult)))
        best_fit_l = min(length_options, key=lambda x: abs(x - target_l))
        effective_atr_mult = data_base_atr * final_mult

        upper_tl.append(tl_data[best_fit_l]["Upper_Trendline"].iloc[i])
        lower_tl.append(tl_data[best_fit_l]["Lower_Trendline"].iloc[i])
        effective_lengths.append(best_fit_l)
        effective_atr_mults.append(effective_atr_mult)

        price = out["Price"].iloc[i]
        curr_atr = out["ATR"].iloc[i]

        upper_break = bool(tl_data[best_fit_l]["Upper_Break"].iloc[i - 1]) if i > 0 else False
        lower_break = bool(tl_data[best_fit_l]["Lower_Break"].iloc[i - 1]) if i > 0 else False

        if lower_break:
            lower_break_streak += 1
        else:
            lower_break_streak = 0
        lower_exit_confirmed = lower_break_streak >= exit_confirm_days

        action = "Hold"
        if units == 0:
            if upper_break:
                units = 1
                entry_price = price
                highest_price = price
                stop_price = price - (curr_atr * effective_atr_mult)
                lower_break_streak = 0
                action = "Buy"
        else:
            highest_price = max(highest_price, price)
            atr_stop = price - (curr_atr * effective_atr_mult)
            pct_trailing_stop = highest_price * (1 - trailing_stop_pct)

            if stop_type == "atr":
                candidate_stop = atr_stop
            elif stop_type == "pct_trailing":
                candidate_stop = pct_trailing_stop
            elif stop_type == "hybrid":
                candidate_stop = max(atr_stop, pct_trailing_stop)
            else:
                raise ValueError("stop_type must be atr, pct_trailing, or hybrid")
            stop_price = max(stop_price, candidate_stop)

            if price <= stop_price or lower_exit_confirmed:
                action = "Stop/Exit" if price <= stop_price else "Trendline Exit"
                units = 0
                entry_price, stop_price, highest_price = np.nan, np.nan, np.nan
                lower_break_streak = 0

        positions.append(units)
        actions.append(action)
        stop_prices.append(stop_price)

    out["Position"] = positions
    out["Action"] = actions
    out["Stop_Price"] = stop_prices
    out["Upper_Trendline"] = upper_tl
    out["Lower_Trendline"] = lower_tl
    out["Effective_Length"] = effective_lengths
    out["Effective_ATR_Mult"] = effective_atr_mults
    out["Asset_Vol_Mult"] = asset_vol_mult
    out["Trade"] = out["Position"].diff().abs().fillna(0)
    out["Strategy_Return"] = (out["Position"].shift(1).fillna(0) * out["Return"]) - (out["Trade"] * TRANSACTION_COST)
    out["Strategy_Cum"] = (1 + out["Strategy_Return"]).cumprod()
    out["BuyHold_Cum"] = (1 + out["Return"]).cumprod()
    return out


price_data_by_ticker = {ticker: download_price_data(ticker, interval=INTERVAL) for ticker in TICKERS}
asset_param_profiles, asset_profile_table = compute_asset_parameter_snapshot(price_data_by_ticker)
print("Latest no-lookahead data-driven parameter snapshot shown for reference.")
display(asset_profile_table.style.format({
    "Annual Vol": "{:.2%}",
    "Trend Efficiency": "{:.4f}",
    "Data Base ATR Mult": "{:.2f}",
}))

stage1_grid = list(product(
    [1.0],
    [1.0],
    [1, 2],
    ["atr", "pct_trailing", "hybrid"],
    [False, True],
))

param_grid_v2 = list(product(
    [0.90, 1.00, 1.10],
    [0.90, 1.00, 1.15],
    [1, 2],
    ["atr", "pct_trailing", "hybrid"],
    [False, True],
))

print(f"Stage 1 combinations: {len(stage1_grid)}")
print(f"Stage 2 combinations: {len(param_grid_v2)}")

all_results_v2 = []
for ticker in TICKERS:
    p_data = price_data_by_ticker[ticker]
    mh_m = performance_metrics(
        p_data.assign(R=p_data["Return"], C=(1 + p_data["Return"]).cumprod()),
        "R",
        "C",
    )
    for len_scale, atr_scale, exit_days, stop_type, use_dynamic in param_grid_v2:
        bt = backtest_vol_strategy_v2(
            p_data,
            len_scale,
            atr_scale,
            asset_vol_mult=1.0,
            use_dynamic_vol=use_dynamic,
            slope_method="stdev",
            exit_confirm_days=exit_days,
            stop_type=stop_type,
            trailing_stop_pct=0.10,
        )
        m = performance_metrics(bt)
        all_results_v2.append({
            "Params": f"LenScale:{len_scale}|ATRScale:{atr_scale}|Exit:{exit_days}|Stop:{stop_type}|Dyn:{use_dynamic}",
            "Length Scale": len_scale,
            "ATR Scale": atr_scale,
            "Stop Type": stop_type,
            "Use Dynamic Vol": use_dynamic,
            "Ticker": ticker,
            "Sharpe": m["Sharpe"],
            "Annual Return": m["Annual Return"],
            "Max Drawdown": m["Max Drawdown"],
            "Trades": int(bt["Trade"].sum()),
            "MH Sharpe": mh_m["Sharpe"],
            "MH Annual Return": mh_m["Annual Return"],
            "MH Max Drawdown": mh_m["Max Drawdown"],
            "Excess Sharpe": m["Sharpe"] - mh_m["Sharpe"],
            "Excess Annual Return": m["Annual Return"] - mh_m["Annual Return"],
            "Drawdown Improvement": m["Max Drawdown"] - mh_m["Max Drawdown"],
        })

result_df = pd.DataFrame(all_results_v2)
ranking_df = result_df.groupby("Params").agg({
    "Sharpe": "mean",
    "Annual Return": "mean",
    "Max Drawdown": "mean",
    "Trades": "mean",
    "MH Sharpe": "mean",
    "MH Annual Return": "mean",
    "MH Max Drawdown": "mean",
    "Excess Sharpe": "mean",
    "Excess Annual Return": "mean",
    "Drawdown Improvement": "mean",
})
ranking_df["Score"] = (
    0.45 * ranking_df["Sharpe"]
    + 0.35 * ranking_df["Annual Return"]
    + 0.20 * ranking_df["Max Drawdown"]
)
ranking_df = ranking_df.sort_values("Score", ascending=False)

stop_type_summary = result_df.groupby(["Stop Type", "Use Dynamic Vol"]).agg({
    "Sharpe": "mean",
    "Annual Return": "mean",
    "Max Drawdown": "mean",
    "Excess Sharpe": "mean",
    "Excess Annual Return": "mean",
    "Drawdown Improvement": "mean",
    "Trades": "mean",
}).sort_values("Sharpe", ascending=False)

print("--- Unified stop type and volatility-adjustment comparison ---")
display(stop_type_summary)
print("--- Global data-driven strategy ranking ---")
display(ranking_df.head(10))

stage1_results = []
for ticker in TICKERS:
    p_data = price_data_by_ticker[ticker]
    mh_m = performance_metrics(
        p_data.assign(R=p_data["Return"], C=(1 + p_data["Return"]).cumprod()),
        "R",
        "C",
    )
    for len_scale, atr_scale, exit_days, stop_type, use_dynamic in stage1_grid:
        bt = backtest_vol_strategy_v2(
            p_data,
            len_scale,
            atr_scale,
            asset_vol_mult=1.0,
            use_dynamic_vol=use_dynamic,
            slope_method="stdev",
            exit_confirm_days=exit_days,
            stop_type=stop_type,
            trailing_stop_pct=0.10,
        )
        m = performance_metrics(bt)
        stage1_results.append({
            "Exit Confirm Days": exit_days,
            "Stop Type": stop_type,
            "Use Dynamic Vol": use_dynamic,
            "Ticker": ticker,
            "Sharpe": m["Sharpe"],
            "Annual Return": m["Annual Return"],
            "Max Drawdown": m["Max Drawdown"],
            "Excess Sharpe": m["Sharpe"] - mh_m["Sharpe"],
            "Excess Annual Return": m["Annual Return"] - mh_m["Annual Return"],
            "Drawdown Improvement": m["Max Drawdown"] - mh_m["Max Drawdown"],
        })

stage1_summary = pd.DataFrame(stage1_results).groupby(["Exit Confirm Days", "Stop Type", "Use Dynamic Vol"]).agg({
    "Sharpe": "mean",
    "Annual Return": "mean",
    "Max Drawdown": "mean",
    "Excess Sharpe": "mean",
    "Excess Annual Return": "mean",
    "Drawdown Improvement": "mean",
}).sort_values("Sharpe", ascending=False)
print("--- Stage 1 exit/stop/vol diagnostic ---")
display(stage1_summary)

BEST_V2_STR = ranking_df.index[0]
p_vals = {k.split(':')[0]: k.split(':')[1] for k in BEST_V2_STR.split('|')}
len_scale = float(p_vals['LenScale'])
atr_scale = float(p_vals['ATRScale'])
exit_days = int(p_vals['Exit'])
stop_type = p_vals['Stop']
use_dynamic_vol = p_vals['Dyn'] == 'True'
print(f"Selected unified data-driven strategy: {BEST_V2_STR}")

best_backtests, final_rows = {}, []
for ticker in TICKERS:
    p_data = price_data_by_ticker[ticker]
    profile = asset_param_profiles[ticker]
    bt = backtest_vol_strategy_v2(
        p_data,
        len_scale,
        atr_scale,
        asset_vol_mult=1.0,
        use_dynamic_vol=use_dynamic_vol,
        slope_method="stdev",
        exit_confirm_days=exit_days,
        stop_type=stop_type,
        trailing_stop_pct=0.10,
    )
    m = performance_metrics(bt)
    mh_m = performance_metrics(
        p_data.assign(R=p_data["Return"], C=(1 + p_data["Return"]).cumprod()),
        "R",
        "C",
    )

    res_row = {
        "Ticker": ticker,
        "Length Scale": len_scale,
        "ATR Scale": atr_scale,
        "Data Base Length": profile["Data Base Length"],
        "Data Base ATR Mult": profile["Data Base ATR Mult"],
        "Length Scale": len_scale,
        "ATR Scale": atr_scale,
        "Annual Vol": profile["Annual Vol"],
        "Trend Efficiency": profile["Trend Efficiency"],
        "Use Dynamic Vol": use_dynamic_vol,
        "Avg Effective Length": bt["Effective_Length"].mean(),
        "Avg Effective ATR Mult": bt["Effective_ATR_Mult"].mean(),
        "Exit Confirm Days": exit_days,
        "Stop Type": stop_type,
        "Sharpe": m["Sharpe"],
        "MH Sharpe": mh_m["Sharpe"],
        "Excess Sharpe": m["Sharpe"] - mh_m["Sharpe"],
        "Total Return": m["Total Return"],
        "MH Total Return": mh_m["Total Return"],
        "Excess Total Return": m["Total Return"] - mh_m["Total Return"],
        "Annual Return": m["Annual Return"],
        "MH Annual Return": mh_m["Annual Return"],
        "Max Drawdown": m["Max Drawdown"],
        "MH Max Drawdown": mh_m["Max Drawdown"],
        "Drawdown Improvement": m["Max Drawdown"] - mh_m["Max Drawdown"],
        "Trades": int(bt["Trade"].sum()),
    }
    final_rows.append(res_row)
    best_backtests[ticker] = (res_row, bt)

best_table = pd.DataFrame(final_rows)

## 6. 找出每個標的最佳策略

In [ ]:
# Display final no-lookahead data-driven strategy performance against MH / buy-and-hold
display(
    best_table.style
    .format("{:.2%}", subset=[
        "Annual Vol", "Total Return", "MH Total Return", "Excess Total Return",
        "Annual Return", "MH Annual Return", "Max Drawdown", "MH Max Drawdown", "Drawdown Improvement"
    ])
    .format("{:.2f}", subset=[
        "Data Base ATR Mult", "Length Scale", "ATR Scale",
        "Trend Efficiency", "Avg Effective Length", "Avg Effective ATR Mult",
        "Sharpe", "MH Sharpe", "Excess Sharpe"
    ])
    .background_gradient(cmap="RdYlGn", subset=["Excess Sharpe"])
)

## 7. Charts: Unified Strategy vs Market Hold

In [ ]:
def calculate_drawdown_series(cumulative_return):
    running_max = cumulative_return.cummax()
    return (cumulative_return / running_max - 1)

# Plot strategy and MH / buy-and-hold performance for each ticker
for ticker in TICKERS:
    if ticker not in best_backtests:
        continue
    row, bt = best_backtests[ticker]

    plt.figure(figsize=(15, 6))
    plt.plot(bt.index, bt["Strategy_Cum"], label=f"Strategy (Total: {row['Total Return']:.2%})", color="#1f77b4", linewidth=2)
    plt.plot(bt.index, bt["BuyHold_Cum"], label=f"MH / Buy-and-Hold (Total: {row['MH Total Return']:.2%})", color="#7f7f7f", alpha=0.65)

    strat_dd = calculate_drawdown_series(bt["Strategy_Cum"])
    mdd_idx = strat_dd.idxmin()
    plt.scatter(mdd_idx, bt.loc[mdd_idx, "Strategy_Cum"], color="red", s=100, zorder=5, label="Strategy MDD")

    len_scale_title = row.get("Length Scale", 1.0)
    atr_scale_title = row.get("ATR Scale", 1.0)
    exit_days = row.get("Exit Confirm Days", "N/A")
    stop_type = row.get("Stop Type", "N/A")
    plt.title(f"{ticker} No-Lookahead Data-Driven Strategy vs MH (LenScale={len_scale_title:.2f}, ATRScale={atr_scale_title:.2f}, Exit={exit_days}, Stop={stop_type})")

    plt.ylabel("Growth of $1")
    plt.legend(loc="upper left")
    plt.grid(True, alpha=0.3)
    plt.show()

summary_compare = best_table.set_index("Ticker")[[
    "Annual Vol", "Trend Efficiency", "Data Base Length", "Data Base ATR Mult",
    "Length Scale", "ATR Scale", "Avg Effective Length", "Avg Effective ATR Mult",
    "Total Return", "MH Total Return", "Excess Total Return",
    "Sharpe", "MH Sharpe", "Excess Sharpe",
    "Max Drawdown", "MH Max Drawdown", "Drawdown Improvement", "Trades"
]].copy()
display(summary_compare.style.format({
    "Annual Vol": "{:.2%}",
    "Trend Efficiency": "{:.4f}",
    "Data Base ATR Mult": "{:.2f}",
    "Length Scale": "{:.2f}",
    "ATR Scale": "{:.2f}",
    "Avg Effective Length": "{:.2f}",
    "Avg Effective ATR Mult": "{:.2f}",
    "Total Return": "{:.2%}",
    "MH Total Return": "{:.2%}",
    "Excess Total Return": "{:.2%}",
    "Sharpe": "{:.2f}",
    "MH Sharpe": "{:.2f}",
    "Excess Sharpe": "{:.2f}",
    "Max Drawdown": "{:.2%}",
    "MH Max Drawdown": "{:.2%}",
    "Drawdown Improvement": "{:.2%}",
}).background_gradient(cmap="RdYlGn", subset=["Excess Sharpe"]))

## 8. 單一標的指標圖：Trendlines with Breaks

In [ ]:
def plot_trendline_breaks(bt, ticker):
    plot_df = bt.tail(500).copy()
    buys = plot_df[plot_df["Action"] == "Buy"]
    sells = plot_df[plot_df["Action"].isin(["Sell", "Stop Loss"])]

    fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True, gridspec_kw={"height_ratios": [3, 1, 1.5]})
    axes[0].plot(plot_df.index, plot_df["Price"], label="Adjusted Price", color="#1f77b4")
    axes[0].plot(plot_df.index, plot_df["Upper_Trendline"], label="Upper Trendline", color="#d62728", linestyle="--")
    axes[0].plot(plot_df.index, plot_df["Lower_Trendline"], label="Lower Trendline", color="#2ca02c", linestyle="--")
    axes[0].scatter(buys.index, buys["Price"], marker="^", color="green", s=70, label="Buy")
    axes[0].scatter(sells.index, sells["Price"], marker="v", color="red", s=70, label="Sell / Stop")
    axes[0].set_title(f"{ticker} Trendlines with Breaks Strategy Signals")
    axes[0].set_ylabel("Price")
    axes[0].legend(loc="upper left")

    axes[1].step(plot_df.index, plot_df["Position"], where="post", color="#9467bd")
    axes[1].set_ylim(-0.05, 1.05)
    axes[1].set_ylabel("Position")
    axes[1].set_title("Position")

    axes[2].plot(plot_df.index, plot_df["Strategy_Cum"], label="Strategy", color="#17becf")
    axes[2].plot(plot_df.index, plot_df["BuyHold_Cum"], label="Market Hold", color="#7f7f7f")
    axes[2].set_ylabel("Growth of $1")
    axes[2].set_title("Cumulative Return")
    axes[2].legend(loc="upper left")
    plt.xlabel("Date")
    plt.tight_layout()
    plt.show()

main_row, main_bt = best_backtests.get(MAIN_TICKER, next(iter(best_backtests.values())))
plot_trendline_breaks(main_bt, MAIN_TICKER)

### 12. 深度分析與優化方向建議

基於全域排名 `ranking_df` 與各標的績效表 `best_table`，我們分析目前的策略瓶頸。

In [ ]:
# 1. Review global stop type and parameter diagnostics
print("Unified stop type comparison:")
display(stop_type_summary)

print("Top 10 global unified strategy parameter sets:")
display(ranking_df.head(10))

print("Stage 1 exit/stop diagnostic:")
display(stage1_summary)

# 2. Average strategy vs MH performance
avg_sharpe = best_table["Sharpe"].mean()
avg_mh_sharpe = best_table["MH Sharpe"].mean()
avg_annual_return = best_table["Annual Return"].mean()
avg_mh_annual_return = best_table["MH Annual Return"].mean()
avg_max_drawdown = best_table["Max Drawdown"].mean()
avg_mh_max_drawdown = best_table["MH Max Drawdown"].mean()
print(f"Average strategy Sharpe: {avg_sharpe:.2f} vs MH {avg_mh_sharpe:.2f}")
print(f"Average annual return: {avg_annual_return:.2%} vs MH {avg_mh_annual_return:.2%}")
print(f"Average max drawdown: {avg_max_drawdown:.2%} vs MH {avg_mh_max_drawdown:.2%}")

# 3. Trading frequency
for ticker, (row, bt) in best_backtests.items():
    trades = bt["Trade"].sum() / 2
    print(f"{ticker} total round-trip trades: {trades:.0f}")

## 9. Train/Test Validation

The notebook downloads maximum daily history and then uses a common analysis window starting from 2000-01-01. This keeps the sample long enough to include multiple market regimes while avoiding very early periods where not every ticker has reliable data.

- Train: 2000-01-01 to 2018-12-31 for parameter selection.
- Test: 2019-01-01 onward for out-of-sample validation.

The objective is the standalone trading strategy performance: Sharpe, annual return, and max drawdown. If train performance is strong but test performance weakens sharply, the parameter set is likely overfit.

In [ ]:
TRAIN_END = "2018-12-31"
TEST_START = "2019-01-01"
WARMUP_START = "2018-01-01"


def slice_backtest_period(bt, start_date):
    sliced = bt.loc[start_date:].copy()
    sliced["Strategy_Cum"] = (1 + sliced["Strategy_Return"]).cumprod()
    sliced["BuyHold_Cum"] = (1 + sliced["Return"]).cumprod()
    return sliced


train_test_rows = []
train_selected_backtests = {}
test_selected_backtests = {}

for ticker in TICKERS:
    full_data = download_price_data(ticker, interval=INTERVAL)
    train_data = full_data.loc[:TRAIN_END].copy()
    test_data_with_warmup = full_data.loc[WARMUP_START:].copy()
    profile = asset_param_profiles[ticker]  # latest snapshot for display only; backtest parameters are rolling/no-lookahead

    best_score = -np.inf
    best_params = None
    best_train_bt = None
    best_train_metrics = None
    best_train_mh = None

    for len_scale_candidate, atr_scale_candidate, exit_days_candidate, candidate_stop_type, candidate_dynamic in param_grid_v2:
        # Keep the final framework unified: train/test uses the same selected stop and dynamic-vol setting.
        if candidate_stop_type != stop_type or candidate_dynamic != use_dynamic_vol:
            continue
        bt_train = backtest_vol_strategy_v2(
            train_data,
            len_scale_candidate,
            atr_scale_candidate,
            asset_vol_mult=1.0,
            use_dynamic_vol=candidate_dynamic,
            slope_method="stdev",
            exit_confirm_days=exit_days_candidate,
            stop_type=candidate_stop_type,
            trailing_stop_pct=0.10,
        )
        m_train = performance_metrics(bt_train)
        mh_train = performance_metrics(
            train_data.assign(R=train_data["Return"], C=(1 + train_data["Return"]).cumprod()),
            "R",
            "C",
        )
        score = 0.45 * m_train["Sharpe"] + 0.35 * m_train["Annual Return"] + 0.20 * m_train["Max Drawdown"]
        if score > best_score:
            best_score = score
            best_params = (len_scale_candidate, atr_scale_candidate, exit_days_candidate, candidate_stop_type, candidate_dynamic)
            best_train_bt = bt_train
            best_train_metrics = m_train
            best_train_mh = mh_train

    selected_len_scale, selected_atr_scale, exit_days, selected_stop_type, selected_dynamic = best_params
    bt_test_full = backtest_vol_strategy_v2(
        test_data_with_warmup,
        selected_len_scale,
        selected_atr_scale,
        asset_vol_mult=1.0,
        use_dynamic_vol=selected_dynamic,
        slope_method="stdev",
        exit_confirm_days=exit_days,
        stop_type=selected_stop_type,
        trailing_stop_pct=0.10,
    )
    bt_test = slice_backtest_period(bt_test_full, TEST_START)
    m_test = performance_metrics(bt_test)
    mh_test = performance_metrics(
        bt_test.assign(R=bt_test["Return"], C=(1 + bt_test["Return"]).cumprod()),
        "R",
        "C",
    )

    train_test_rows.append({
        "Ticker": ticker,
        "Data Base Length": profile["Data Base Length"],
        "Data Base ATR Mult": profile["Data Base ATR Mult"],
        "Length Scale": selected_len_scale,
        "ATR Scale": selected_atr_scale,
        "Annual Vol": profile["Annual Vol"],
        "Trend Efficiency": profile["Trend Efficiency"],
        "Use Dynamic Vol": selected_dynamic,
        "Exit Confirm Days": exit_days,
        "Stop Type": selected_stop_type,
        "Train Sharpe": best_train_metrics["Sharpe"],
        "Train MH Sharpe": best_train_mh["Sharpe"],
        "Train Excess Sharpe": best_train_metrics["Sharpe"] - best_train_mh["Sharpe"],
        "Test Sharpe": m_test["Sharpe"],
        "Test MH Sharpe": mh_test["Sharpe"],
        "Test Excess Sharpe": m_test["Sharpe"] - mh_test["Sharpe"],
        "Train Annual Return": best_train_metrics["Annual Return"],
        "Train MH Annual Return": best_train_mh["Annual Return"],
        "Test Annual Return": m_test["Annual Return"],
        "Test MH Annual Return": mh_test["Annual Return"],
        "Train Max Drawdown": best_train_metrics["Max Drawdown"],
        "Train MH Max Drawdown": best_train_mh["Max Drawdown"],
        "Test Max Drawdown": m_test["Max Drawdown"],
        "Test MH Max Drawdown": mh_test["Max Drawdown"],
    })
    train_selected_backtests[ticker] = best_train_bt
    test_selected_backtests[ticker] = bt_test

train_test_table = pd.DataFrame(train_test_rows)
display(
    train_test_table.style
    .format("{:.2f}", subset=[
        "Data Base ATR Mult", "Length Scale", "ATR Scale",
        "Trend Efficiency", "Train Sharpe", "Train MH Sharpe", "Train Excess Sharpe",
        "Test Sharpe", "Test MH Sharpe", "Test Excess Sharpe"
    ])
    .format("{:.2%}", subset=[
        "Annual Vol", "Train Annual Return", "Train MH Annual Return", "Test Annual Return", "Test MH Annual Return",
        "Train Max Drawdown", "Train MH Max Drawdown", "Test Max Drawdown", "Test MH Max Drawdown"
    ])
    .background_gradient(cmap="RdYlGn", subset=["Test Excess Sharpe"])
)

## 9. Result Interpretation and Improvement Direction

The strategy now uses data-driven base parameters. Instead of giving every ticker the same starting `Length` and `ATR Mult`, the notebook measures each asset's annualized volatility and trend efficiency through rolling past-only windows.

The intuition is simple. A high-volatility asset usually needs a longer breakout window and a wider stop because normal price noise is larger. A smoother trending asset can use a shorter window because its breakouts are less likely to be random noise. The final strategy remains unified because these differences come from the same formula across all tickers.

This is different from per-ticker optimization. Per-ticker optimization asks, "Which parameter worked best for this ticker in history?" The data-driven method asks, "Given this ticker's volatility and trend structure, what base parameter should the same rule assign?" The second approach is easier to explain and has lower overfitting risk.

Potential improvement directions:

1. **Check whether data-driven bases improve test Sharpe**: the train/test table is the most important validation. If train improves but test weakens, the mapping is too aggressive.
2. **Tune the mapping formula, not each ticker**: adjust the exponents or clip ranges for `Length Mult` and `ATR Mult Factor` instead of manually choosing values per asset.
3. **Keep dynamic volatility optional**: if `Use Dynamic Vol=True` does not improve test results, keep it disabled.
4. **Add an entry-quality filter**: volume confirmation or ATR breakout buffer may reduce false breakouts more effectively than more stop tuning.
5. **Add cost and slippage sensitivity**: higher trade counts need stricter validation after costs.

## 10. 生成式 AI 應用與反思

本作業使用生成式 AI 協助將 TradingView 指標概念轉換為 Python 回測流程。為了提高程式品質，我先要求 AI 查清楚指標的核心特徵，例如 pivot-based trendlines、breakout signals，以及 ATR、Stdev、Linreg 三種斜率方法，再要求它避免未來函數問題。

實作過程中特別需要注意 pivot 指標的延遲。若直接用當天 pivot high/low 產生訊號，回測會偷看到未來價格，導致結果過度樂觀。因此 notebook 中設定 pivot 必須等待右側 `length` 根 K 線後才確認，交易也使用前一日訊號在下一日執行。

未來可以進一步加入 walk-forward validation，把前半段資料用於尋找參數，後半段資料用於驗證，降低資料探勘造成的過度配適風險。

## 11. Conclusion

This notebook implements a Trendlines with Breaks style breakout strategy with multi-asset backtesting, market-hold comparison, visualization, and train/test validation.

The latest strategy uses a unified rule across all tickers: fixed base parameters are adjusted by volatility multipliers instead of optimizing a separate parameter set for every asset. This keeps the strategy easier to explain and reduces overfitting risk while still allowing high-volatility and low-volatility assets to be treated differently in a systematic way.

The final result should be evaluated by comparing strategy metrics with MH over the same period. A useful strategy does not need to win on every metric, but it should show a clear tradeoff, such as better Sharpe or smaller drawdown without giving up too much annual return. Before using the strategy outside this assignment, transaction costs, slippage, and additional out-of-sample testing should be added.